# **Network Analysis**

Three networks from one topic run's `posts_<ISSUES>.parquet` (every post, duplicates kept, with the topic of
its text), following the flowchart:

```
posts + topic ─┬─ PEP ──────── dedup → clean_for_bert → NER → name normalization → PEP nodes (+ sentiment) → co-mention edges ─┐
               └─ Author ──┬── influence:    @mention / RT take-up → nodes → edges ───────────────────────────────────────────┼─> network statistics
                           └── coordination: same text / same target in one time window → evidence → nodes → edges ───────────┘
```

Set `NETWORKS` to the branches you need; the others are skipped. Every network gets the same summary
statistics, its own extras, a Gephi export of **all** its nodes and edges, and topics on its edges and
communities. A comparison table closes the notebook.

Merged from `author_network_extractor.ipynb` (influence, coordination) and `pep_ner_simple.ipynb` (PEP).
Transfer entropy is not carried over.

### **Mounting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### **Configuration**

In [ ]:
import datetime as dt, os

ROBJ        = 'Putusan MK Gibran' # research object: figure titles and the output folder name
POSTS_FILE  = '/content/drive/MyDrive/model_result/Putusan MK Gibran_20102026/posts_Putusan MK Gibran.parquet' # posts_<ISSUES>.parquet from topic_modelling
RESULT_PATH = '/content/drive/MyDrive/model_result/Network Analysis' # results go to <ROBJ>_<first>_<last> in here
NETWORKS    = ['pep', 'influence', 'coordination'] # branches to run; leave one out to skip it

# account resolution (influence, coordination)
ALIASES    = {}        # canonical key -> canonical key, for merges the rules miss, e.g. {'republikaonline': 'republikacoid'}
# coordination
WINDOW     = '30min'   # co-occurrence time bucket
MIN_CO     = 3         # distinct shared items before a pair gets an edge
MIN_WORDS  = 5         # shorter texts are too generic to count as shared
MAX_GROUP  = 50        # skip buckets with more accounts than this (viral posts, not campaigns)
# PEP
NER_MODEL  = 'cahya/bert-base-indonesian-NER'
NER_BATCH  = 64
NER_MAXLEN = 256       # tokens read per document: actors are named in the opening, the tail is runtime
NER_MIN_P  = 0.85      # span confidence floor; below it the PER class is mostly noise
PEP_WINDOW = 60        # chars either side of a name searched for a jabatan cue
MAX_NAMES  = 12        # documents naming more people are roundups, not relationships: no co-mention edges
# figures (drawings only; exports always hold every node and edge)
TOP_FIG    = 300       # nodes drawn in network figures
TOP_N      = 20        # persons in the PEP tables and figures
SEED       = 42

### **Notebook Set Up**
The PEP branch also needs `torch` and `transformers`, which ship with Colab.

In [ ]:
%pip install -q "great[all] @ git+https://github.com/azmkto/GREAT-Tools.git"

import re
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

from great import is_media_account, sent_class, sent_colors
from great.text import clean_for_bert
from great.viz import checks, prep
from great.viz.style import apply_style

apply_style()
SENTS  = ['positive', 'neutral', 'negative']
SLOTS  = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']  # categorical, fixed order
OTHER  = '#b8b7b0'
TONE_CMAP = LinearSegmentedColormap.from_list(
    'tone', [sent_colors['negative'], sent_colors['neutral'], sent_colors['positive']])
AUTHOR_NETWORKS = bool({'influence', 'coordination'} & set(NETWORKS))
SUMMARIES = {}   # one network_summary() per branch that ran, for the comparison table

### **Accessing File**
`prep.prepare_data` keeps the time of day in `Date`, which the coordination windows need. A `great` older than
this change floors it to midnight; the cell warns if every post sits at 00:00.

In [ ]:
posts = pd.read_parquet(POSTS_FILE)
df1, start_date, end_date = prep.prepare_data(posts)
for c in ('Followers', 'Retweeted', 'Favourited'):    # stored as text when the export mixed numbers and text
    if c in df1:
        df1[c] = pd.to_numeric(df1[c], errors='coerce')
if (df1['Date'] == df1['Date'].dt.normalize()).all():
    print('WARNING: every Date is at 00:00 -- update great; coordination windows need the time of day')

TOPIC_NAME = df1.dropna(subset=['Topic']).drop_duplicates('Topic').set_index('Topic')['TopicLabel']
STAMP = f"{df1['Date'].min():%d%m%Y}_{df1['Date'].max():%d%m%Y}"
OUT_DIR = f'{RESULT_PATH}/{ROBJ}_{STAMP}'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'{start_date} - {end_date}   {len(df1):,} posts, {df1.Mentions.nunique():,} distinct texts, '
      f'{df1.Topic.notna().mean():.1%} with a topic, {len(TOPIC_NAME)} topics   ->   {OUT_DIR}')
checks.frame_info(df1, 'Posts')

### **Shared helpers**
- `communities(U)`: Louvain on an undirected weighted graph, ids ordered by size so colours stay put across runs.
- `network_summary(G, U, member, Q, name)`: the statistics every network reports (below).
- `topic_mix(rows, keys)`: per group (an edge, a community), the dominant topic and the top three as `label (posts)`.
- `export_gephi(nodes, edges, name)`: every node and edge to CSV. Gephi: *Data Laboratory → Import Spreadsheet*,
  nodes first (`Id`, `Label`), then edges (`Source`, `Target`, `Type`, `Weight`). It checks that every edge endpoint is a node.

| Statistic | Meaning |
|---|---|
| `Nodes`, `Edges` | size of the network |
| `Density` | edges present / edges possible |
| `Components` | connected pieces (weakly connected for the directed influence network); an isolated PEP is its own piece |
| `Giant component share` | share of nodes in the largest piece |
| `Average clustering` | how often two neighbours of a node are also linked (unweighted, on the undirected graph) |
| `Communities`, `Modularity Q` | Louvain partition. $Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$: weight inside communities beyond a degree-preserving random graph; $Q \gtrsim 0.3$ is usually read as clear structure |

In [ ]:
def communities(U):
    # Louvain on an undirected weighted graph; ids ordered by size so colours stay put across runs.
    comms = sorted(nx.community.louvain_communities(U, weight='Weight', seed=SEED), key=len, reverse=True)
    member = pd.Series({n: i for i, c in enumerate(comms) for n in c}, dtype='int64')
    return member, (nx.community.modularity(U, comms, weight='Weight') if U.number_of_edges() else 0.0)


def network_summary(G, U, member, Q, name):
    # G: the network as built (directed or not); U: its undirected weighted version.
    n = G.number_of_nodes()
    comps = list(nx.connected_components(U))
    return pd.Series({
        'Nodes': n,
        'Edges': G.number_of_edges(),
        'Density': nx.density(G) if n > 1 else 0.0,
        'Components': len(comps),
        'Giant component share': max(map(len, comps)) / n if n else 0.0,
        'Average clustering': nx.average_clustering(U) if n else 0.0,
        'Communities': member.nunique(),
        'Modularity Q': Q,
    }, name=name)


def topic_mix(rows, keys, k=3, names=None):
    # Per group of `keys`: dominant topic, its label, and the top `k` topics as 'label (posts)'.
    names = TOPIC_NAME if names is None else names
    c = (rows.dropna(subset=['Topic']).groupby(keys + ['Topic']).size().rename('n').reset_index()
             .sort_values('n', ascending=False, kind='stable').groupby(keys).head(k))
    c['s'] = c['Topic'].map(names).fillna(c['Topic'].astype(str)) + ' (' + c['n'].astype(str) + ')'
    g = c.groupby(keys)
    out = pd.DataFrame({'Topic': g['Topic'].first(), 'Topics': g['s'].agg(', '.join)})
    out['TopicLabel'] = out['Topic'].map(names)
    return out[['Topic', 'TopicLabel', 'Topics']]


def export_gephi(nodes, edges, name):
    # Every node and edge, Gephi's column names first.
    nodes = nodes.rename_axis('Id').reset_index()
    nodes = nodes[['Id', 'Label'] + [c for c in nodes.columns if c not in ('Id', 'Label')]]
    edges = edges[['Source', 'Target', 'Type', 'Weight'] + [c for c in edges.columns if c not in ('Source', 'Target', 'Type', 'Weight')]]
    missing = (set(edges.Source) | set(edges.Target)) - set(nodes.Id)
    assert not missing, f'{name}: {len(missing)} edge endpoints are not in the node table, e.g. {sorted(missing)[:3]}'
    nodes.to_csv(f'{OUT_DIR}/{name}_nodes.csv', index=False)
    edges.to_csv(f'{OUT_DIR}/{name}_edges.csv', index=False)
    print(f'{name}: {len(nodes):,} nodes, {len(edges):,} edges -> {name}_nodes.csv, {name}_edges.csv')


def plot_distributions(table, cols, title, top=5):
    # Log-binned density (top) and CCDF P(X >= x) (bottom) per column, top nodes named on the CCDF.
    fig, axes = plt.subplots(2, len(cols), figsize=(3.4 * len(cols), 6), squeeze=False)
    for j, col in enumerate(cols):
        x = table[col][table[col] > 0].sort_values(ascending=False)
        hist, ccdf = axes[0, j], axes[1, j]
        if x.empty:
            hist.set_title(f'{col} (all zero)')
            continue
        bins = np.logspace(np.log10(x.min()), np.log10(x.max()), 25) if x.max() > x.min() else 5
        hist.hist(x, bins=bins, density=True, color=SLOTS[0], edgecolor='white', linewidth=0.5)
        hist.set(xscale='log', yscale='log', title=col, ylabel='p(x)' if j == 0 else None)
        p = np.arange(1, len(x) + 1) / len(x)
        ccdf.plot(x, p, ls='none', marker='o', ms=2.5, color=SLOTS[0])
        ccdf.set(xscale='log', yscale='log', xlabel=col, ylabel='P(X ≥ x)' if j == 0 else None)
        for k, (idx, val) in enumerate(x.head(top).items()):
            ccdf.annotate(table.at[idx, 'Label'], (val, p[k]), xytext=(-6, 10 + 11 * k), textcoords='offset points',
                          ha='right', fontsize=6, arrowprops=dict(arrowstyle='-', lw=0.4, color='0.4'))
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def draw_network(G, nodes, size_col, title, color_col='Community', tone=False, top=TOP_FIG, n_labels=10, file=None):
    # Giant component, top `top` nodes by `size_col`; colour by `color_col` (8 slots + Other), edges by Tone if asked.
    parts = nx.weakly_connected_components(G) if G.is_directed() else nx.connected_components(G)
    giant = nodes.loc[list(max(parts, key=len))]
    H = G.subgraph(giant.nlargest(top, size_col).index)
    sub = nodes.loc[list(H)]
    cats = sorted(sub[color_col].unique())[:len(SLOTS)]
    colour = {c: SLOTS[i] for i, c in enumerate(cats)}
    size = 10 + 290 * np.sqrt(sub[size_col] / sub[size_col].max())
    pos = nx.spring_layout(H, seed=SEED)

    fig, ax = plt.subplots(figsize=(10, 10))
    nx.draw_networkx_edges(H, pos, ax=ax, width=0.4, alpha=0.5, arrows=False,
                           edge_color=[H.edges[e]['Tone'] for e in H.edges] if tone else OTHER,
                           edge_cmap=TONE_CMAP if tone else None, edge_vmin=-1, edge_vmax=1)
    nx.draw_networkx_nodes(H, pos, ax=ax, node_size=size.to_numpy(), linewidths=0.5, edgecolors='white',
                           node_color=[colour.get(c, OTHER) for c in sub[color_col]])
    nx.draw_networkx_labels(H, pos, ax=ax, font_size=7, labels=sub.nlargest(n_labels, size_col).Label.to_dict())
    handles = [Line2D([], [], ls='none', marker='o', color=colour[c], label=f'{color_col} {c}') for c in cats]
    if sub[color_col].nunique() > len(cats):
        handles.append(Line2D([], [], ls='none', marker='o', color=OTHER, label='Other'))
    ax.legend(handles=handles, loc='upper left', fontsize=7, frameon=False)
    if tone:
        fig.colorbar(plt.cm.ScalarMappable(cmap=TONE_CMAP, norm=plt.Normalize(-1, 1)),
                     ax=ax, shrink=0.4, label='Edge tone (negative → positive)')
    ax.set_title(title)
    ax.axis('off')
    if file:
        fig.savefig(f'{OUT_DIR}/{file}', dpi=200, bbox_inches='tight')
    plt.show()


toy = pd.DataFrame({'a': ['x', 'x', 'x', 'y'], 'Topic': pd.array([1, 1, 2, pd.NA], dtype='Int64')})
toy_names = pd.Series({1: 'Banjir', 2: 'Pajak'})
assert topic_mix(toy, ['a'], names=toy_names).loc['x'].tolist() == [1, 'Banjir', 'Banjir (2), Pajak (1)']
assert 'y' not in topic_mix(toy, ['a'], names=toy_names).index      # a group with no topic gets none, not a crash
toy_g = nx.Graph([('a', 'b'), ('b', 'c')]); toy_g.add_node('d')
toy_s = network_summary(toy_g, toy_g, pd.Series({'a': 0, 'b': 0, 'c': 0, 'd': 1}), 0.0, 'toy')
assert (toy_s['Components'], toy_s['Giant component share'], toy_s['Communities']) == (2, 0.75, 2)

## **1. Account Resolution** (influence, coordination)
One outlet shows up under several spellings: `www.kompastv.com` as a news author, `@KOMPASTVCOM` when someone
mentions it, `nasional.kompas.com` as a subdomain. Every author and every mentioned handle goes through
`canonical()` so they share one key:

1. lowercase, strip `http(s)://`, `www.`, `@` and any `/path`;
2. domain-shaped values keep only the registered domain (`nasional.kompas.com` → `kompas.com`; `x.co.id` keeps three labels);
3. drop everything that is not `a-z0-9` (the same normalisation `great.is_media_account` uses);
4. apply `ALIASES`.

Check the merge report; a wrong merge is fixed by adding the key to `ALIASES`.

**Take-up links** are built here because both author branches use them. A post by B that contains `@A` (a mention,
reply, quote or `RT @A` retweet) is read as *B took up A*. They are read from the raw `Mentions`, since
`clean_for_bert` strips the handles.

In [ ]:
TWO_PART_SLD = {'co', 'or', 'ac', 'go', 'web', 'my', 'sch', 'net', 'com'}
HANDLE = re.compile(r'(?<![\w.])@([\w.]*\w)')        # lookbehind skips e-mail addresses
RT_OF  = re.compile(r'^\s*RT @([\w.]*\w)')


def canonical(name):
    # Merge key for an account: '@KOMPASTVCOM', 'www.kompastv.com' -> 'kompastvcom'.
    s = re.sub(r'^(https?://)?(www\.)?@?', '', str(name).strip().lower()).split('/')[0]
    parts = s.split('.')
    if len(parts) > 2 and re.fullmatch(r'[a-z]{2,3}', parts[-1]):   # domain, not an Instagram-style dotted handle
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r'[^a-z0-9]', '', ''.join(parts))
    return ALIASES.get(key, key)


def influence_links(posts):
    # One row per (post, distinct mentioned account): Source = mentioned A, Key = author B.
    text = posts['Mentions'].astype(str)
    links = posts.assign(
        Source=text.map(lambda t: sorted({canonical(h) for h in HANDLE.findall(t)})),
        RT=text.str.extract(RT_OF, expand=False).map(canonical, na_action='ignore'),
    ).explode('Source').dropna(subset=['Source'])
    links = links[links.Source != links.Key]
    return links.assign(Retweet=links.Source == links.RT)


def node_table(edges):
    # Account attributes for every endpoint of `edges`; mentioned-only handles get Posts = 0.
    ids = pd.unique(edges[['Source', 'Target']].to_numpy().ravel())
    nodes = accounts.reindex(ids).rename_axis('Id')
    nodes['Label'] = nodes.Label.fillna(nodes.index.to_series())
    nodes['IsMedia'] = [is_media_account(i) if pd.isna(m) else m for i, m in nodes.IsMedia.items()]
    nodes['Posts'] = nodes.Posts.fillna(0).astype(int)
    return nodes


assert canonical('@KOMPASTVCOM') == canonical('www.kompastv.com') == 'kompastvcom'
assert canonical('nasional.kompas.com') == canonical('@kompascom') == 'kompascom'
assert canonical('https://www.republika.co.id/berita/x') == 'republikacoid'
assert canonical('prabowo.subianto') == canonical('@Prabowo_Subianto') == 'prabowosubianto'

In [ ]:
if AUTHOR_NETWORKS:
    df1['Key'] = df1['Author'].dropna().map(canonical)
    known = df1.dropna(subset=['Key'])

    accounts = known.groupby('Key').agg(
        Label=('Author', lambda s: s.mode().iat[0]),       # most common raw spelling
        Posts=('Author', 'size'),
        Followers=('Followers', 'max'),
        Media=('Media', lambda s: s.mode().iat[0]),
    )
    accounts['IsMedia'] = [is_media_account(l, m) for l, m in zip(accounts.Label, accounts.Media)]

    links = influence_links(df1.dropna(subset=['Key', 'Mentions']))
    merges = known.groupby('Key').agg(Spellings=('Author', 'unique'), Rows=('Author', 'size'))
    merges = merges[merges.Spellings.map(len) > 1].sort_values('Rows', ascending=False)
    print(f'{df1.Author.nunique():,} raw authors -> {len(accounts):,} accounts; {len(merges):,} keys merged 2+ spellings; '
          f'{len(links):,} take-up links')
    display(merges.head(30))

## **2. Influence Network**
**Edges** run **A → B** (B took up A), the direction influence flows. Self-links after merging are dropped.

| Column | Meaning |
|---|---|
| `Weight` | posts in which B took up A |
| `Retweets` | of those, how many were `RT @A` |
| `positive / neutral / negative` | those posts split by their sentiment label |
| `Tone` | $(n_+ - n_-)/\text{Weight} \in [-1, 1]$ |
| `Dominant` | most frequent label |
| `Topic`, `TopicLabel`, `Topics` | dominant topic of those posts, and the top three |

**Tone is not stance.** The sentiment label belongs to B's *post*, not to B's attitude toward A. Report Tone as
*the tone of discourse around A*, never as *B opposes A*.

**Extras on the nodes**: `Reach` (out-degree, distinct accounts that took A up), `WeightedReach`, `Amplifies`
(in-degree), `PageRank` (random walk following attention B → A: influence that propagates), `Eigenvector`
(undirected, giant component; 0 outside it). Betweenness is left to Gephi (*Statistics → Network Diameter*); it is
O(nm) and slow in pure Python at this size.

In [ ]:
def influence_edges(links):
    g = links.groupby(['Source', 'Key'])
    edges = g['Sentiment'].value_counts().unstack(fill_value=0).reindex(columns=SENTS, fill_value=0)
    edges['Weight'] = g.size()
    edges['Retweets'] = g['Retweet'].sum()
    edges['Tone'] = (edges.positive - edges.negative) / edges.Weight
    edges['Dominant'] = edges[SENTS].idxmax(axis=1)
    edges['Type'] = 'Directed'
    edges = edges.rename_axis(index=['Source', 'Target'], columns=None).reset_index()
    return edges[['Source', 'Target', 'Type', 'Weight', 'Retweets', *SENTS, 'Tone', 'Dominant']]


toy = pd.DataFrame({
    'Key':       ['bob', 'bob', 'carol', 'alice', 'dave'],
    'Mentions':  ['RT @Alice: mantap @alice', '@alice jelek', 'halo @bob.', 'saya @alice', 'mail redaksi@kompas.com'],
    'Sentiment': ['positive', 'negative', 'neutral', 'positive', 'neutral'],
})
t = influence_edges(influence_links(toy)).set_index(['Source', 'Target'])
assert (t.loc[('alice', 'bob'), ['Weight', 'Retweets', 'positive', 'negative', 'Tone']] == [2, 1, 1, 1, 0]).all()
assert set(t.index) == {('alice', 'bob'), ('bob', 'carol')}     # self-mention and e-mail dropped
assert (t[SENTS].sum(axis=1) == t.Weight).all()

In [ ]:
if 'influence' in NETWORKS:
    edges = influence_edges(links)
    edges = edges.join(topic_mix(links.rename(columns={'Key': 'Target'}), ['Source', 'Target']), on=['Source', 'Target'])

    G = nx.from_pandas_edgelist(edges, 'Source', 'Target', edge_attr=['Weight', 'Tone'], create_using=nx.DiGraph)
    pair = np.sort(edges[['Source', 'Target']].to_numpy(), axis=1)       # undirected: sum A->B and B->A
    und = edges.assign(u=pair[:, 0], v=pair[:, 1]).groupby(['u', 'v'], as_index=False).Weight.sum()
    U = nx.from_pandas_edgelist(und, 'u', 'v', edge_attr='Weight')
    giant = U.subgraph(max(nx.connected_components(U), key=len))

    nodes = node_table(edges)
    nodes['Reach']         = pd.Series(dict(G.out_degree()))
    nodes['WeightedReach'] = pd.Series(dict(G.out_degree(weight='Weight')))
    nodes['Amplifies']     = pd.Series(dict(G.in_degree()))
    nodes['PageRank']      = pd.Series(nx.pagerank(G.reverse(copy=False), weight='Weight'))
    nodes['Eigenvector']   = pd.Series(nx.eigenvector_centrality_numpy(giant, weight='Weight')).reindex(nodes.index, fill_value=0)
    nodes['Community'], Q  = communities(U)
    assert abs(nodes.PageRank.sum() - 1) < 1e-6

    summary = network_summary(G, U, nodes.Community, Q, 'Influence')
    SUMMARIES['Influence'] = summary
    extras = pd.Series({'Reciprocity': nx.reciprocity(G),
                        **{f'Tone share {s}': edges[s].sum() / edges.Weight.sum() for s in SENTS}}, name='Influence')
    print(f'{len(links):,} take-up links -> {len(edges):,} weighted edges')
    display(pd.concat([summary, extras]).to_frame())

In [ ]:
if 'influence' in NETWORKS:
    for col in ['PageRank', 'Eigenvector', 'Reach']:
        print(f'\nTop 10 by {col}')
        display(nodes.nlargest(10, col)[['Label', 'PageRank', 'Eigenvector', 'Reach', 'Posts', 'Followers', 'IsMedia', 'Community']])

### **Camps**
Per community: size, the five highest-PageRank accounts, share of news/outlet accounts, the topics its members
take others up on, and two tones:
- **Tone inside**: posts where both A and B are in the camp.
- **Tone outward**: posts where the author B is in the camp and the mentioned A is outside, i.e. how the camp talks about outsiders.

In [ ]:
if 'influence' in NETWORKS:
    camp_edges = edges.assign(cA=edges.Source.map(nodes.Community), cB=edges.Target.map(nodes.Community))
    tone = lambda g: (g.positive - g.negative) / g.Weight

    camps = nodes.groupby('Community').agg(Size=('Label', 'size'), MediaShare=('IsMedia', 'mean'))
    camps['Top accounts'] = nodes.sort_values('PageRank', ascending=False).groupby('Community').Label.agg(lambda s: ', '.join(s.head(5)))
    camps['Top topics']   = topic_mix(links.assign(C=links.Key.map(nodes.Community)), ['C'])['Topics']
    camps['Tone inside']  = tone(camp_edges[camp_edges.cA == camp_edges.cB].groupby('cB')[['positive', 'negative', 'Weight']].sum())
    camps['Tone outward'] = tone(camp_edges[camp_edges.cA != camp_edges.cB].groupby('cB')[['positive', 'negative', 'Weight']].sum())
    camps.to_csv(f'{OUT_DIR}/influence_camps.csv')
    display(camps.head(15))

### **Distributions and figure**
Log-binned densities and complementary CDFs on log–log axes. A straight-looking CCDF tail suggests a heavy tail but is
**not** evidence of a power law; that needs a maximum-likelihood fit with likelihood-ratio tests against lognormal
and similar alternatives.

In [ ]:
if 'influence' in NETWORKS:
    plot_distributions(nodes, ['Reach', 'WeightedReach', 'Amplifies', 'PageRank', 'Eigenvector'],
                       f'Influence network statistics, {ROBJ} ({start_date} - {end_date})')
    draw_network(G, nodes, 'PageRank', f'Influence network, top {TOP_FIG} accounts by PageRank', tone=True,
                 file='influence_network.png')

In [ ]:
if 'influence' in NETWORKS:
    export_gephi(nodes, edges, 'influence')

## **3. Coordination Network** (experimental, internal)
Coordinated accounts rarely mention each other; they *behave* alike. Two accounts are linked when, inside the same
`WINDOW` bucket, they

- post the **same text** (normalised: lowercase, no URLs, @mentions, `RT` prefix or punctuation; at least `MIN_WORDS` words), or
- take up the **same target** (mention or retweet the same account).

One shared item proves nothing, since thousands retweet the same viral post. A pair gets an edge only after sharing
at least `MIN_CO` **distinct** items, and buckets with more than `MAX_GROUP` accounts are skipped as virality rather
than campaigns. News rows and outlet accounts are excluded, because syndicated wire copy is legitimate duplication.

**Internal use only.** An edge here is a lead to inspect, not a finding. Read the evidence table before believing any cluster.

### **3a. Feasibility probe**
If `Type` shows no retweet/quote rows, retweeters are not in the export and the same-target signal is weak. If
almost no social text is shared by 2+ accounts, same-text will find nothing.

In [ ]:
def normalize_text(t):
    t = re.sub(r'^\s*rt @[\w.]+:?', ' ', str(t).lower())
    t = re.sub(r'http\S+|www\.\S+|@[\w.]+', ' ', t)
    return ' '.join(re.sub(r'[^\w\s]', ' ', t).split())


if 'coordination' in NETWORKS:
    social = df1[(df1.Media != 'News') & df1.Key.notna() & ~df1.Key.map(accounts.IsMedia).eq(True)]
    social = social.assign(Text=social.Mentions.map(normalize_text))

    if 'Type' in df1:
        display(pd.crosstab(df1.Type, df1.Media))
    rt_share = social.Mentions.astype(str).str.match(r'\s*RT @').mean()
    print(f"Social rows: {len(social):,}  |  starting with 'RT @': {rt_share:.1%}")

    long_texts = social[social.Text.str.split().str.len() >= MIN_WORDS]
    shared = long_texts.groupby('Text').agg(Accounts=('Key', 'nunique'), Posts=('Key', 'size'),
                                            Spread=('Date', lambda t: t.max() - t.min()))
    shared = shared[shared.Accounts >= 2].sort_values('Accounts', ascending=False)
    print(f'Texts posted by 2+ accounts: {len(shared):,}')
    display(shared.Spread.describe())
    display(shared.head(10))

### **3b. Co-occurrence edges**
`# ponytail:` fixed time buckets (`Date.floor(WINDOW)`) instead of a sliding window, so two posts a minute apart
across a bucket boundary are missed. Switch to a sort-and-scan window if that matters.

Each edge gets the topics of the posts behind its shared items (`Topic`, `TopicLabel`, `Topics`).

In [ ]:
def co_occurrence(events, min_co=MIN_CO):
    # events: Key, Item, Bucket. Pairs sharing >= min_co distinct Items in the same bucket; plus the evidence rows.
    ev = events[['Key', 'Item', 'Bucket']].drop_duplicates()
    groups = ev.groupby(['Item', 'Bucket']).Key.agg(sorted)
    big = (groups.map(len) > MAX_GROUP).sum()
    groups = groups[groups.map(len).between(2, MAX_GROUP)]
    rows = [(a, b, item, bucket) for (item, bucket), keys in groups.items() for a, b in combinations(keys, 2)]
    evidence = pd.DataFrame(rows, columns=['Source', 'Target', 'Item', 'Bucket'])
    weight = evidence.groupby(['Source', 'Target']).Item.nunique().rename('Weight')
    print(f'  {big:,} buckets over MAX_GROUP skipped; {(weight < min_co).sum():,} pairs under MIN_CO dropped')
    weight = weight[weight >= min_co]
    evidence = evidence.join(weight, on=['Source', 'Target'], how='inner')
    return weight.reset_index(), evidence.drop(columns='Weight')


t0 = pd.Timestamp('2026-08-01 10:00')
toy = pd.DataFrame({'Key':  ['a', 'b', 'a', 'b', 'a', 'b', 'a', 'c'],
                    'Item': ['x', 'x', 'y', 'y', 'z', 'z', 'w', 'w'],
                    'Bucket': [t0, t0] * 3 + [t0, t0 + pd.Timedelta('2h')]})
w, ev = co_occurrence(toy)
assert w.values.tolist() == [['a', 'b', 3]] and len(ev) == 3     # a-c never share a bucket
assert co_occurrence(toy, min_co=4)[0].empty

In [ ]:
if 'coordination' in NETWORKS:
    social_links = links[links.Key.isin(social.Key.unique())]
    text_events = long_texts.assign(Item=long_texts.Text, Bucket=long_texts.Date.dt.floor(WINDOW))[['Key', 'Item', 'Bucket', 'Topic']]
    targ_events = social_links.assign(Item=social_links.Source, Bucket=social_links.Date.dt.floor(WINDOW))[['Key', 'Item', 'Bucket', 'Topic']]
    print('Same text:');   text_w, text_ev = co_occurrence(text_events)
    print('Same target:'); targ_w, targ_ev = co_occurrence(targ_events)

    coord_edges = (pd.concat([text_w.assign(Kind='TextCo'), targ_w.assign(Kind='TargetCo')])
                   .pivot_table(index=['Source', 'Target'], columns='Kind', values='Weight', fill_value=0)
                   .reindex(columns=['TextCo', 'TargetCo'], fill_value=0)
                   .rename_axis(columns=None).reset_index())
    coord_edges['Weight'] = coord_edges.TextCo + coord_edges.TargetCo
    coord_edges['Type'] = 'Undirected'
    coord_evidence = pd.concat([text_ev.assign(Kind='text'), targ_ev.assign(Kind='target')], ignore_index=True)

    # topics of the posts behind each shared item: the Source account's posts of that item in that bucket
    ev_topics = coord_evidence.merge(pd.concat([text_events, targ_events]).rename(columns={'Key': 'Source'}),
                                     on=['Source', 'Item', 'Bucket'])
    if not coord_edges.empty:
        coord_edges = coord_edges.join(topic_mix(ev_topics, ['Source', 'Target']), on=['Source', 'Target'])

    print(f'Same-text pairs: {len(text_w):,}  |  same-target pairs: {len(targ_w):,}  |  coordination edges: {len(coord_edges):,}')
    if coord_edges.empty:
        print('No pair reached MIN_CO. Nothing coordinated is detectable at this threshold; the cells below skip.')

### **3c. Clusters and evidence**
Extras: `Degree`, `WeightedDegree` per account; per cluster its members, followers, top shared texts and topics.

In [ ]:
if 'coordination' in NETWORKS and not coord_edges.empty:
    CG = nx.from_pandas_edgelist(coord_edges, 'Source', 'Target', edge_attr='Weight')
    coord_nodes = node_table(coord_edges)
    coord_nodes['Degree']         = pd.Series(dict(CG.degree()))
    coord_nodes['WeightedDegree'] = pd.Series(dict(CG.degree(weight='Weight')))
    coord_nodes['Community'], coord_Q = communities(CG)
    SUMMARIES['Coordination'] = network_summary(CG, CG, coord_nodes.Community, coord_Q, 'Coordination')

    top_texts = (coord_evidence[coord_evidence.Kind == 'text']
                 .assign(C=lambda d: d.Source.map(coord_nodes.Community))
                 .groupby('C').Item.agg(lambda s: ' | '.join(s.value_counts().index[:3])))
    clusters = coord_nodes.groupby('Community').agg(Size=('Label', 'size'), Followers=('Followers', 'sum'))
    clusters['Members'] = (coord_nodes.sort_values('WeightedDegree', ascending=False)
                           .groupby('Community').Label.agg(lambda s: ', '.join(s.head(10))))
    clusters['Top shared texts'] = top_texts
    clusters['Top topics'] = topic_mix(ev_topics.assign(C=ev_topics.Source.map(coord_nodes.Community)), ['C'])['Topics']
    clusters.to_csv(f'{OUT_DIR}/coordination_clusters.csv')
    display(SUMMARIES['Coordination'].to_frame())
    display(clusters.head(15))
    display(coord_evidence.sort_values(['Source', 'Target', 'Bucket']).head(30))

In [ ]:
if 'coordination' in NETWORKS and not coord_edges.empty:
    plot_distributions(coord_nodes, ['Degree', 'WeightedDegree'], 'Coordination network: accounts')
    plot_distributions(coord_edges.assign(Label=coord_edges.Source + ' – ' + coord_edges.Target),
                       ['Weight', 'TextCo', 'TargetCo'], 'Coordination network: pairs')
    draw_network(CG, coord_nodes, 'WeightedDegree', 'Coordination network', file='coordination_network.png')

In [ ]:
if 'coordination' in NETWORKS and not coord_edges.empty:
    export_gephi(coord_nodes, coord_edges, 'coordination')
    coord_evidence.to_csv(f'{OUT_DIR}/coordination_evidence.csv', index=False)

## **4. PEP Co-mention Network**
Politically exposed persons named in the posts:

```
dedup on Mentions → clean_for_bert → NER (PER spans + char offsets) → name normalization → jabatan cue near span → PEP → co-mention
```

The posts are **deduplicated first**, with a leading `RT @handle:` removed: an article and its 300 retweets name
the same people once, not 301 times.
`clean_for_bert` removes URLs and @handles but **keeps case, punctuation and titles** (Presiden, Menteri, Pak…):
the NER model uses them as context and `role_of()` reads them. Titles are stripped only from the extracted
name strings. From here on a document is identified by its position `row` in `docs`.

In [ ]:
if 'pep' in NETWORKS:
    # a retweet carries the article behind 'RT @handle:', so the prefix goes before deduplicating
    unrt = df1['Mentions'].astype(str).str.replace(r'^\s*RT @[\w.]+:?\s*', '', regex=True)
    pep_docs = df1.assign(Text=unrt).drop_duplicates(subset='Text').reset_index(drop=True)
    docs = pep_docs['Text'].map(clean_for_bert).tolist()
    print(f'{len(df1):,} posts -> {len(docs):,} distinct documents')

### **4a. NER: find person spans**
Token classification with `aggregation_strategy='simple'` merges B-/I- word pieces into one span with character
offsets. Truncation is set on the tokenizer (`model_max_length`); the pipeline refuses `truncation=` as a call
argument. The `[:NER_MAXLEN * 6]` slice stops the tokenizer reading a 20k-char article it would truncate anyway;
it is a prefix, so offsets still point into `docs[row]`. The name is sliced from the text rather than taken from
`e['word']`, so the original casing survives. The spans are saved: this is the slow part.

In [ ]:
if 'pep' in NETWORKS:
    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

    tok = AutoTokenizer.from_pretrained(NER_MODEL, model_max_length=NER_MAXLEN)
    ner = pipeline('token-classification', model=AutoModelForTokenClassification.from_pretrained(NER_MODEL),
                   tokenizer=tok, aggregation_strategy='simple', device=0 if torch.cuda.is_available() else -1)

In [ ]:
if 'pep' in NETWORKS:
    CHUNK = 10_000   # print progress every CHUNK documents
    texts = [d[:NER_MAXLEN * 6] for d in docs]
    rows = []
    for row, ents in enumerate(ner(texts, batch_size=NER_BATCH)):
        for e in ents:
            if e['entity_group'].upper().startswith('PER') and e['score'] >= NER_MIN_P:
                rows.append((row, texts[row][e['start']:e['end']], e['start'], e['end']))
        if (row + 1) % CHUNK == 0 or row + 1 == len(texts):
            print(f'  {row + 1:,}/{len(texts):,}  spans so far {len(rows):,}')
    spans = pd.DataFrame(rows, columns=['row', 'surface', 'start', 'end'])
    spans.to_parquet(f'{OUT_DIR}/pep_ner_spans.parquet', index=False)
    print(f"{len(spans):,} PER spans in {spans['row'].nunique():,} documents, {spans['surface'].nunique():,} surface forms")

### **4b. Name normalization: surface form → one canonical person**
Three steps, all on the span strings only:
1. `canon()` strips leading jabatan/honorifics and trailing gelar ('Bapak Presiden Prabowo' → 'Prabowo').
2. `PEP_ALIASES` maps press short forms to full names. Written by hand, the corpus can't infer it; extend it.
3. `build_fold()` merges a one-word name into the multi-word name containing it, **only if one candidate dominates**
   (≥ 2× the runner-up). A bare 'Yudhoyono' with SBY and AHY both frequent stays unmerged rather than joining two people.

In [ ]:
HONORIFIC = (r'(?:Bapak|Bpk|Pak|Ibu|Bu|Saudara|Sdr|Prof|Dr|Drs|Ir|Hj|KH|Kiai|Gus|'
             r'Jenderal|Jendral|Letjen|Mayjen|Brigjen|Laksamana|Marsekal|Komjen|Irjen|'
             r'Kombes|AKBP|Pol|Polisi|Tuan|Mr|Mrs|Mas|Mbak|Mba|Habib|Bang|Kang)\b\.?')
JABATAN = (r'(?:Presiden|Wakil Presiden|Wapres|Menteri|Menko|Wakil Menteri|Wamen|Gubernur|'
           r'Wakil Gubernur|Bupati|Wali ?Kota|Walikota|Ketua Umum|Ketum|Ketua|Wakil Ketua|'
           r'Sekjen|Sekretaris Jenderal|Kapolri|Kapolda|Kapolres|Panglima|Jaksa Agung|Hakim|'
           r'Komisioner|Anggota|Juru Bicara|Jubir|Kepala|Direktur|Dirjen)\b')
GELAR = r'\b(?:SH|MH|MBA|MSi|SE|SIP|PhD|LLM|MSc|ST|MM|SPd|MPd)\b\.?'


def canon(name):
    # Surface form -> bare personal name, '' when nothing survives.
    s = re.sub(r'\s+', ' ', str(name).replace('##', '')).strip(' .,:;-"\'')
    for _ in range(3):                      # 'Jenderal Polisi Drs X' needs three passes
        s = re.sub(rf'^{JABATAN}\s*', '', s, flags=re.I)
        s = re.sub(rf'^{HONORIFIC}\s*', '', s, flags=re.I)
    s = re.sub(GELAR, ' ', s, flags=re.I)
    s = re.sub(r"[^A-Za-z'\- ]", ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    if len(s) < 3 or len(s.split()) > 5:    # a 6-word 'name' is a sentence the tagger swallowed
        return ''
    return ' '.join(w.upper() if (w.isupper() and len(w) <= 3) else w.title() for w in s.split())


PEP_ALIASES = {
    'prabowo': 'Prabowo Subianto',        'gibran': 'Gibran Rakabuming Raka',
    'jokowi': 'Joko Widodo',              'megawati': 'Megawati Soekarnoputri',
    'sby': 'Susilo Bambang Yudhoyono',    'ahy': 'Agus Harimurti Yudhoyono',
    'luhut': 'Luhut Binsar Pandjaitan',   'airlangga': 'Airlangga Hartarto',
    'sri mulyani': 'Sri Mulyani Indrawati', 'bahlil': 'Bahlil Lahadalia',
    'zulhas': 'Zulkifli Hasan',           'cak imin': 'Muhaimin Iskandar',
    'muhaimin': 'Muhaimin Iskandar',      'anies': 'Anies Baswedan',
    'ganjar': 'Ganjar Pranowo',           'puan': 'Puan Maharani',
    'tito': 'Tito Karnavian',             'yusril': 'Yusril Ihza Mahendra',
    'sjafrie': 'Sjafrie Sjamsoeddin',     'dasco': 'Sufmi Dasco Ahmad',
    'amran': 'Andi Amran Sulaiman',       'purbaya': 'Purbaya Yudhi Sadewa',
}
AMBIGUOUS = {'muhammad', 'ahmad', 'budi', 'bambang', 'hadi', 'andi', 'agus', 'joko', 'andre'}
PEP_ALIASES |= dict.fromkeys(AMBIGUOUS, '')


def build_fold(counts, min_ratio=2.0):
    # One-word name -> the dominant multi-word name containing it.
    longs = sorted(((n, c) for c, n in counts.items() if len(c.split()) > 1), reverse=True)
    out = {}
    for c in counts:
        if len(c.split()) != 1:
            continue
        cand = [(n, l) for n, l in longs if c.lower() in {w.lower() for w in l.split()}]
        if cand and (len(cand) == 1 or cand[0][0] >= min_ratio * cand[1][0]):
            out[c] = cand[0][1]
    return out


assert canon('Jenderal Polisi Listyo Sigit Prabowo') == 'Listyo Sigit Prabowo'
assert canon('Bapak Presiden Prabowo') == 'Prabowo'
assert build_fold({'Prabowo': 5, 'Prabowo Subianto': 20}) == {'Prabowo': 'Prabowo Subianto'}

if 'pep' in NETWORKS:
    name = spans['surface'].map(lambda s: PEP_ALIASES.get(canon(s).lower(), canon(s)))
    fold = build_fold(Counter(name[name != '']))
    spans['Person'] = name.map(lambda p: fold.get(p, p))
    spans = spans[spans['Person'] != ''].reset_index(drop=True)
    print(f"{name.nunique():,} names -> {spans['Person'].nunique():,} persons ({len(fold)} folded)")
    display(spans['Person'].value_counts().head(15))

### **4c. Who is a PEP?**
For every span, look `PEP_WINDOW` characters either side for a jabatan cue. A window, not the whole document: in
'Menteri X berbicara, sementara Z menolak', Z must not inherit X's title. `ROLE_CUES` is ordered most-specific first
('Wakil Presiden' before 'Presiden'), and the first cue that matches wins. Each person gets the **majority** role over
all their spans; ties go to the more specific cue. `PEP_SEED` covers people so well known that the press stops writing
their office next to the name.

In [ ]:
ROLE_CUES = {
    'Wakil Presiden':     r'Wakil Presiden|Wapres',
    'Presiden':           r'Presiden(?!\s+(?:Direktur|Komisaris))',
    'Menteri':            r'Menteri|Menko|Wakil Menteri|Wamen',
    'Kepala Lembaga':     r'Kepala (?:Badan|Staf|BIN|BPS|Bappenas|Otorita)|'
                          r'Ketua (?:KPK|KPU|OJK|BPK|Bawaslu|Mahkamah)|Komisioner',
    'Ketua Partai':       r'Ketua Umum|Ketum|Sekretaris Jenderal (?:Partai|PDI|Golkar|Gerindra)',
    'Anggota Parlemen':   r'anggota (?:DPR|DPD|MPR|DPRD|Komisi|dewan)|Ketua DPR|Ketua MPR|'
                          r'Ketua DPD|Wakil Ketua DPR|legislator|Ketua Komisi',
    'Gubernur':           r'Gubernur|Pj\.? Gubernur',
    'Bupati / Wali Kota': r'Bupati|Wali ?Kota|Walikota',
    'TNI / Polri':        r'Kapolri|Kapolda|Kapolres|Panglima|KSAD|KSAL|KSAU|Kabareskrim|'
                          r'Jenderal|Irjen|Komjen|Kombes',
    'Penegak Hukum':      r'Jaksa Agung|Kejaksaan|Jaksa|Hakim|penyidik KPK|Ketua Mahkamah',
    'Pemuka Agama':       r'\b(?:Habib|Gus|Ustadz|Ustaz|Kiai|KH)\b',
}
ROLE_PATS = {r: re.compile(p, re.I) for r, p in ROLE_CUES.items()}
PEP_SEED = {
    'Prabowo Subianto', 'Gibran Rakabuming Raka', 'Joko Widodo',
    'Megawati Soekarnoputri', 'Susilo Bambang Yudhoyono', 'Agus Harimurti Yudhoyono',
    'Puan Maharani', 'Anies Baswedan', 'Ganjar Pranowo', 'Muhaimin Iskandar',
    'Surya Paloh', 'Zulkifli Hasan', 'Airlangga Hartarto', 'Sri Mulyani Indrawati',
}


def role_of(row, start, end):
    w = docs[row][max(0, start - PEP_WINDOW):end + PEP_WINDOW]
    return next((r for r, p in ROLE_PATS.items() if p.search(w)), None)


if 'pep' in NETWORKS:
    spans['RoleHit'] = [role_of(r, s, e) for r, s, e in zip(spans['row'], spans['start'], spans['end'])]
    vote = (spans.dropna(subset=['RoleHit'])
            .groupby(['Person', 'RoleHit']).size().rename('n').reset_index()
            .assign(order=lambda d: d['RoleHit'].map(list(ROLE_CUES).index))
            .sort_values(['n', 'order'], ascending=[False, True])
            .drop_duplicates('Person').set_index('Person')['RoleHit'])

    people = spans['Person'].value_counts().rename('Spans').to_frame()
    people['Role'] = people.index.map(vote)
    people.loc[people['Role'].isna() & people.index.isin(PEP_SEED), 'Role'] = 'Seed list'
    people['PEP'] = people['Role'].notna()
    spans = spans.join(people[['Role', 'PEP']], on='Person')

    print(f"{people['PEP'].sum():,} PEP of {len(people):,} persons")
    print('not PEP (sample):', list(people.index[~people['PEP']][:15]))
    display(people[people['PEP']].head(20))

### **4d. PEP nodes: ranking**
One row per (document, person): a text naming Prabowo four times counts as **one** mention. `Share %` = percentage of
all documents that name the person. This table is the node list: **every** PEP, not only the top `TOP_N`.

In [ ]:
if 'pep' in NETWORKS:
    pep = (spans.loc[spans['PEP'], ['row', 'Person', 'Role']]
           .drop_duplicates(['row', 'Person'])
           .join(pep_docs[['Date', 'Media', 'Sentiment', 'Topic']], on='row'))
    pep.to_parquet(f'{OUT_DIR}/pep_mentions.parquet', index=False)

    rank = (pep.groupby('Person')
            .agg(Role=('Role', 'first'), Mentions=('row', 'size'), Platforms=('Media', 'nunique'),
                 First_seen=('Date', 'min'), Last_seen=('Date', 'max'))
            .sort_values('Mentions', ascending=False))
    rank['Share %'] = (100 * rank['Mentions'] / len(docs)).round(2)
    top = rank.index[:TOP_N].tolist()
    display(rank.head(TOP_N))
    print('Most common surface forms of the top persons:')
    display(spans.groupby('Person')['surface'].agg(lambda s: s.value_counts().head(5).to_dict()).loc[top])

### **4e. Daily diffusion heatmap**
Share = % of that day's documents naming the person. The denominator is **all** documents of the day, so a quiet day
where 2 of 2 texts name Prabowo doesn't show as 100 %. Each row is scaled to its own peak (otherwise Prabowo sets the
colour scale) and rows are sorted by peak day. A short bright block is a burst actor who showed up for one event; a
continuous band is a fixture of the discourse.

In [ ]:
if 'pep' in NETWORKS:
    per_day = pep_docs['Date'].dt.to_period('D').value_counts().sort_index()
    counts_d = pd.crosstab(pep['Date'].dt.to_period('D'), pep['Person'])[top].reindex(per_day.index, fill_value=0)
    share = counts_d.div(per_day, axis=0) * 100
    intensity = share / share.max()
    order = intensity.idxmax().sort_values().index

    fig, ax = plt.subplots(figsize=(14, 0.4 * len(order) + 2))
    im = ax.imshow(intensity[order].T.values, aspect='auto', cmap='magma', vmin=0, vmax=1)
    ax.set_yticks(range(len(order)), order)
    ax.set_xticks(range(len(share)), share.index.strftime('%d\n%b'), fontsize=9, rotation=45)
    ax.set_title('Politically exposed persons: daily share of documents (row-normalised)', fontweight='bold')
    fig.colorbar(im, ax=ax, pad=0.01, shrink=0.7, label='share / person peak share')
    plt.tight_layout()
    fig.savefig(f'{OUT_DIR}/pep_diffusion.png', dpi=200, bbox_inches='tight')
    plt.show()
    display(share[order].round(2).T)

### **4f. Sentiment toward PEP**
Proportions, not counts: Prabowo's raw bar would be ten times longer than everyone else's and hide the comparison.
`n` is shown so a 100 %-negative n = 3 person isn't over-read. The label belongs to the document, so read it as
*the tone of documents naming the person*, not as a measured attitude toward them.

In [ ]:
if 'pep' in NETWORKS:
    sent = pd.crosstab(pep['Person'], pep['Sentiment'])
    cols = [s for s in sent_class if s in sent.columns] + [s for s in sent.columns if s not in sent_class]
    pct = sent[cols].div(sent.sum(axis=1), axis=0) * 100
    pct.round(1).assign(n=sent.sum(axis=1)).loc[rank.index].to_csv(f'{OUT_DIR}/pep_sentiment.csv')

    shown = pct.loc[top[::-1]]
    ax = shown.plot.barh(stacked=True, width=0.75, figsize=(11, 0.4 * len(shown) + 2),
                         color=[sent_colors.get(s, '#999999') for s in cols])
    for i, n in enumerate(sent.sum(axis=1).loc[top[::-1]]):
        ax.text(101, i, f'n={n:,}', va='center', fontsize=9)
    ax.set_xlim(0, 100); ax.set_xlabel("% of that person's mentions"); ax.set_ylabel('')
    ax.set_title('Sentiment toward PEP', fontweight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=len(cols), frameon=False)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/pep_sentiment.png', dpi=200, bbox_inches='tight')   # before show(), which clears the figure
    plt.show()

### **4g. Co-mention edges and statistics**
An edge means two PEPs are named in the same document, weighted by how many documents do this; its topics come from
those documents. Documents naming more than `MAX_NAMES` people are skipped: a list of 40 names is a news roundup, not a
relationship. Every PEP is a node, including those never named alongside another (`Degree` = 0).

Extras on the nodes: `Degree` (distinct co-mentioned PEPs), `Strength` (co-mention documents), `Betweenness`
(share of shortest paths through the person, unweighted: who brokers between otherwise separate groups of elites).

In [ ]:
if 'pep' in NETWORKS:
    names_per_doc = pep.groupby('row')['Person'].agg(lambda s: sorted(set(s)))
    roundups = names_per_doc.map(len) > MAX_NAMES
    pair_rows = pd.DataFrame([(a, b, row) for row, names in names_per_doc[~roundups].items()
                              for a, b in combinations(names, 2)], columns=['Source', 'Target', 'row'])
    pair_rows['Topic'] = pair_rows['row'].map(pep_docs['Topic'])
    print(f'{roundups.sum():,} documents naming more than {MAX_NAMES} PEPs skipped '
          f'({names_per_doc[roundups].map(lambda n: len(n) * (len(n) - 1) // 2).sum():,} pairs not counted)')

    pep_edges = pair_rows.groupby(['Source', 'Target']).size().rename('Weight').reset_index()
    pep_edges['Type'] = 'Undirected'
    if not pep_edges.empty:
        pep_edges = pep_edges.join(topic_mix(pair_rows, ['Source', 'Target']), on=['Source', 'Target'])

    PG = nx.Graph()
    PG.add_nodes_from(rank.index)                                      # every PEP, isolated ones too
    PG.add_weighted_edges_from(pep_edges[['Source', 'Target', 'Weight']].itertuples(index=False), weight='Weight')

    pep_nodes = rank.copy()
    pep_nodes.insert(0, 'Label', pep_nodes.index)
    pep_nodes['Degree']      = pd.Series(dict(PG.degree()))
    pep_nodes['Strength']    = pd.Series(dict(PG.degree(weight='Weight')))
    pep_nodes['Betweenness'] = pd.Series(nx.betweenness_centrality(PG))
    pep_nodes['Community'], pep_Q = communities(PG)
    SUMMARIES['PEP co-mention'] = network_summary(PG, PG, pep_nodes.Community, pep_Q, 'PEP co-mention')

    display(pd.concat([SUMMARIES['PEP co-mention'],
                       pd.Series({'Isolated PEPs': int((pep_nodes.Degree == 0).sum())})]).to_frame('PEP co-mention'))
    display(pep_nodes.nlargest(TOP_N, 'Betweenness')[['Role', 'Mentions', 'Degree', 'Strength', 'Betweenness', 'Community']])
    display(pep_edges.sort_values('Weight', ascending=False).head(20))

In [ ]:
if 'pep' in NETWORKS and not pep_edges.empty:
    draw_network(PG, pep_nodes, 'Mentions', 'PEP co-mention network', top=TOP_FIG, n_labels=TOP_N,
                 file='pep_network.png')

In [ ]:
if 'pep' in NETWORKS:
    export_gephi(pep_nodes, pep_edges, 'pep')

## **5. Comparison**
The shared statistics of every network that ran, side by side. Density and clustering compare directly; node and
edge counts reflect what each network counts (accounts vs. persons), so read them as scale, not as more or less structure.

In [ ]:
comparison = pd.concat(SUMMARIES.values(), axis=1)
comparison.to_csv(f'{OUT_DIR}/network_comparison.csv')
print('Saved to', OUT_DIR)
comparison